# C2 - SWELL-KW INSPECTION notebook (step 6)

**Purpose:** only LOOK at the real SWELL-KW files. Nothing is cleaned, deleted, merged or trained here. SWELL-KW is **not mixed** with StudentLife.

## What SWELL-KW is (from the dataset page)
25 people did typical knowledge work (writing reports, making presentations, reading e-mail, searching) in a lab. Their working conditions were manipulated with two stressors: **email interruptions** and **time pressure**. Computer logging, facial expressions, body posture and physiology (heart rate, HRV, skin conductance) were recorded. Participants also rated **task load (NASA-TLX), mental effort (RSME), emotion (SAM) and perceived stress** for each condition.

## Files inspected (only the C2-relevant ones)
| File | Why |
|---|---|
| `A - Computer interaction features (Ulog - All Features per minute)-Sheet_1.csv` | keyboard/mouse/app behaviour per minute (behavioural signals) |
| `D - Physiology features (HR_HRV_SCL - final).csv` | heart rate, HRV, skin conductance per minute |
| `Behavioral-features - per minute.xlsx` | per-minute merged behavioural features (172 variables) |
| `Questionnaire Blok Results_final - performance added.xlsx` | self-reported task load / effort / stress per block (candidate TARGETS) + performance |
| `SWELL-KW - overview available data.xlsx` | which data exist for which participants |

Everything here is **exploration only**. Self-report ratings would only be TARGETS / reference, never model inputs (C2 design).

## Cell 1 - Connect Drive and find the SWELL files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

TARGET_NAME = 'Behavioral-features - per minute.xlsx'
SWELL_DIR = None
for dirpath, dirnames, filenames in os.walk('/content/drive/MyDrive'):
    if TARGET_NAME in filenames:
        SWELL_DIR = Path(dirpath)
        break
assert SWELL_DIR is not None, 'SWELL files not found in Drive - check the upload.'

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('SWELL folder:', SWELL_DIR)
for f in sorted(SWELL_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

## Cell 2 - Overview file (what data exist for whom)
`SWELL-KW - overview available data.xlsx` is small, so it is printed in full.

In [ ]:
ov_path = SWELL_DIR / 'SWELL-KW - overview available data.xlsx'
xl = pd.ExcelFile(ov_path)
print('sheets:', xl.sheet_names)
for sh in xl.sheet_names:
    df = xl.parse(sh)
    print(f'\n=== sheet "{sh}": {df.shape[0]} rows x {df.shape[1]} columns')
    print(df.to_string(max_rows=90))

## Cell 3 - File A: computer interaction features (per minute)
Real columns, participants (`PP`), blocks (`Blok`), conditions (`Condition`), and missing values.

In [ ]:
A_path = SWELL_DIR / 'A - Computer interaction features (Ulog - All Features per minute)-Sheet_1.csv'
A = pd.read_csv(A_path)
A = A.loc[:, ~A.columns.str.startswith('Unnamed')]          # drop empty trailing columns (the file ends with ",,")
print('shape:', A.shape)
print('columns:', list(A.columns))
print('\nparticipants (PP):', A['PP'].nunique(), '->', sorted(A['PP'].unique().tolist()))
print('blocks (Blok):', A['Blok'].value_counts().sort_index().to_dict())
print('conditions:', A['Condition'].value_counts().to_dict())
print('\nfirst rows:')
print(A.head(4).to_string())
print('\nfraction missing (columns with any missing):')
miss = A.isna().mean()
print(miss[miss > 0].round(3).to_string() if (miss > 0).any() else 'none')
print('\nrows (minutes) per participant: min/median/max =', A.groupby('PP').size().min(), A.groupby('PP').size().median(), A.groupby('PP').size().max())

## Cell 4 - File D: physiology features (per minute)
Heart rate (`HR`), HRV (`RMSSD`) and skin conductance (`SCL`). Block column is called `C` here.

In [ ]:
D_path = SWELL_DIR / 'D - Physiology features (HR_HRV_SCL - final).csv'
D = pd.read_csv(D_path)
D = D.loc[:, ~D.columns.str.startswith('Unnamed')]
print('shape:', D.shape)
print('columns:', list(D.columns))
print('\nparticipants (PP):', D['PP'].nunique(), '->', sorted(D['PP'].unique().tolist()))
print('blocks (C):', D['C'].value_counts().sort_index().to_dict())
print('conditions:', D['Condition'].value_counts().to_dict())
print('\nfirst rows:')
print(D.head(4).to_string())
print('\nsummary of HR / RMSSD / SCL:')
print(D[['HR', 'RMSSD', 'SCL']].describe().round(2).to_string())
print('\nfraction missing:')
print(D.isna().mean().round(3).to_string())

## Cell 5 - File: Behavioral-features per minute (172 variables)
Too many columns to read one by one, so we group them by name prefix and look at the key identifier columns.

In [ ]:
B_path = SWELL_DIR / 'Behavioral-features - per minute.xlsx'
xl = pd.ExcelFile(B_path)
print('sheets:', xl.sheet_names)
B = xl.parse(xl.sheet_names[0])
B = B.loc[:, ~B.columns.astype(str).str.startswith('Unnamed')]
print('shape:', B.shape)
cols = [str(c) for c in B.columns]
print('\nfirst 12 columns:', cols[:12])
prefix = pd.Series([re.match(r'[A-Za-z]+', c).group(0) if re.match(r'[A-Za-z]+', c) else c for c in cols])
print('\ncolumn name prefixes (count):')
print(prefix.value_counts().head(25).to_string())
print('\ncolumns that look like identifiers / labels:')
print([c for c in cols if re.search(r'(?i)^pp$|participant|condition|blok|block|label|stress|time|workload|effort|mental', c)])
print('\nfirst 3 rows (first 14 columns):')
print(B.iloc[:3, :14].to_string())

## Cell 6 - Questionnaire results (candidate targets)
All column names are printed, because this is where the self-report ratings (task load, mental effort, stress, emotion) and the performance column live.

In [ ]:
Q_path = SWELL_DIR / 'Questionnaire Blok Results_final - performance added.xlsx'
xl = pd.ExcelFile(Q_path)
print('sheets:', xl.sheet_names)
Q = xl.parse(xl.sheet_names[0])
Q = Q.loc[:, ~Q.columns.astype(str).str.startswith('Unnamed')]
print('shape:', Q.shape)
print('\nALL columns:')
for i, c in enumerate(Q.columns):
    print(f'{i:2d}  {c}')
print('\nfirst 5 rows:')
print(Q.head(5).to_string())
print('\ncolumns with missing values:')
m = Q.isna().sum()
print(m[m > 0].to_string() if (m > 0).any() else 'none')

## Cell 7 - Do the files line up? (participants and conditions)
Checks that A (computer interaction) and D (physiology) describe the same participants and conditions, and how many minutes each participant has per condition. This is only a check, **no merge yet**.

In [ ]:
pa, pd_ = set(A['PP'].unique()), set(D['PP'].unique())
print('participants in A only:', sorted(pa - pd_), '| in D only:', sorted(pd_ - pa), '| in both:', len(pa & pd_))
print('conditions A:', sorted(A['Condition'].unique().tolist()), '| D:', sorted(D['Condition'].unique().tolist()))

tab_A = A.groupby(['PP', 'Condition']).size().unstack(fill_value=0)
tab_D = D.groupby(['PP', 'Condition']).size().unstack(fill_value=0)
print('\nminutes per participant and condition (File A), first 12 participants:')
print(tab_A.head(12).to_string())
print('\nminutes per participant and condition (File D), first 12 participants:')
print(tab_D.head(12).to_string())
print('\ntimestamp examples  A:', A['timestamp'].head(3).tolist(), '| D:', D['timestamp'].head(3).tolist())

## Cell 8 - Save small summaries

In [ ]:
summary = pd.DataFrame({
    'file': ['A computer interaction', 'D physiology', 'Behavioral per minute', 'Questionnaire'],
    'rows': [len(A), len(D), len(B), len(Q)],
    'columns': [A.shape[1], D.shape[1], B.shape[1], Q.shape[1]],
})
summary.to_csv(OUT_DIR / 'swell_inspection_summary.csv', index=False)
pd.DataFrame({'questionnaire_columns': [str(c) for c in Q.columns]}).to_csv(OUT_DIR / 'swell_questionnaire_columns.csv', index=False)
pd.DataFrame({'behavioral_columns': [str(c) for c in B.columns]}).to_csv(OUT_DIR / 'swell_behavioral_columns.csv', index=False)
print(summary.to_string(index=False))
print('saved in', OUT_DIR)

## Cell 9 - What to send back to Claude
Nothing: Claude reads the outputs of Cells 1-8 directly from this Colab tab.